# Benchmarking

This benchmarking compares the following models:
- MViTv2_S_16x4 
- MViTv2_B_32x3

each at 16 and 32 frames (models with interpolated positional encodings have suffix _e for extended and _r for reduced), with batch sizes:
- train: 1 - 8
- inference: 1 - 16

where the models OOM on the final batch size.

During training, a stable GPU clock speed range of 1910 - 1930 MHz was recorded, so the GPU clock speed is locked with a slight margin at 1900 with the command:
```bash
sudo nvidia-smi -lgc 1900
```
and can be reset with:
```bash
sudo nvidia-smi -rgc
```

The following commands were used to run this set of experiments:



In [1]:
import json

#locals
# from code.run_types import ResSet, RunRes
import pandas as pd

from src.run_types import RESULTS_DIR

In [2]:
def load_mvit_benchmarks(benchmark_path):
    """
    Load the shared all_benchmark.json, keep only MViTv2 variants, drop OOM runs,
    map model names, and return separate DataFrames for training and inference.
    """
    with open(benchmark_path, "r") as f:
        raw = json.load(f)

    runs = raw["runs"]

    # Model name mapping
    model_name_map = {
        # "MViTv2_S": "MViTv2\\_S",
        "MViTv2_B_32x3": "MViTv2\\_B\\_32x3",
        "MViTv2_B_32x3_r": "MViTv2\\_B\\_32x3",
        # "MViTv2_S_e": "MViTv2\\_S\\_e",
        "MViTv2_S_16x4": "MViTv2\\_S\\_16x4",
        "MViTv2_S_16x4_e": "MViTv2\\_S\\_16x4\\_e"
    }

    records = []

    for run in runs.values():
        arch = run.get("arch", "")
        if arch not in model_name_map:
            continue
        if "error" in run:
            continue

        config = run["config"]
        results = run["results"]

        records.append({
            "model": model_name_map[arch],  # mapped name
            "num_frames": config["num_frames"],
            "mode": "Train" if config.get("full_step", False) else "Infer",
            "batch_size": config["batch_size"],

            "gpu_util_mean": results["gpu_utilisation_percent"]["mean"],
            "gpu_util_std": results["gpu_utilisation_percent"]["std"],

            "latency_ms_mean": results["latency_ms"]["mean"],
            "latency_ms_std": results["latency_ms"]["std"],

            "throughput_samp_per_s_mean": results["throughput_samples_per_s"]["mean"],
            "throughput_samp_per_s_std": results["throughput_samples_per_s"]["std"],

            "peak_mem_mb_mean": results["peak_memory_mb"]["mean"],
            "peak_mem_mb_std": results["peak_memory_mb"]["std"],
        })

    df = pd.DataFrame(records)

    metric_cols = [
        "gpu_util_mean", "gpu_util_std",
        "latency_ms_mean", "latency_ms_std",
        "throughput_samp_per_s_mean", "throughput_samp_per_s_std",
        "peak_mem_mb_mean", "peak_mem_mb_std",
    ]
    df[metric_cols] = df[metric_cols].round(2)

    # Sort by num_frames, then batch_size
    train_df = df[df["mode"] == "Train"].sort_values(
        ["num_frames", "latency_ms_mean","batch_size"]
    ).reset_index(drop=True)
    infer_df = df[df["mode"] == "Infer"].sort_values(
        ["num_frames","latency_ms_mean", "batch_size"]
    ).reset_index(drop=True)

    return train_df, infer_df

## Load and display results

In [3]:
train_df, infer_df = load_mvit_benchmarks(RESULTS_DIR / "all_benchmark.json")

print("Training DataFrame:")
display(train_df)

print("\nInference DataFrame:")
display(infer_df)

Training DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S\_16x4,16,Train,1,98.81,0.03,224.06,0.06,4.46,0.00,1800.72,0.0
1,MViTv2\_B\_32x3,16,Train,1,98.73,0.11,319.51,0.79,3.13,0.01,2598.79,0.0
2,MViTv2\_S\_16x4,16,Train,2,99.56,0.01,430.89,0.43,4.64,0.00,3293.91,0.0
3,MViTv2\_B\_32x3,16,Train,2,99.40,0.02,613.38,0.13,3.26,0.00,4759.60,0.0
4,MViTv2\_S\_16x4,16,Train,4,99.83,0.01,828.60,0.48,4.83,0.00,6223.95,0.0
5,MViTv2\_B\_32x3,16,Train,4,99.83,0.00,1174.99,0.09,3.40,0.00,9070.12,0.0
6,MViTv2\_S\_16x4\_e,32,Train,1,99.75,0.01,574.10,0.05,1.74,0.00,3950.84,0.0
7,MViTv2\_B\_32x3,32,Train,1,99.72,0.06,807.67,0.17,1.24,0.00,5599.70,0.0
8,MViTv2\_S\_16x4\_e,32,Train,2,99.89,0.00,1124.22,0.04,1.78,0.00,7572.60,0.0
9,MViTv2\_B\_32x3,32,Train,2,99.86,0.02,1582.92,0.03,1.26,0.00,10733.43,0.0



Inference DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S\_16x4,16,Infer,1,98.00,0.00,54.16,0.12,18.46,0.04,468.83,0.0
1,MViTv2\_B\_32x3,16,Infer,1,98.00,0.00,76.25,0.20,13.11,0.03,532.39,0.0
2,MViTv2\_S\_16x4,16,Infer,2,99.00,0.00,103.88,0.06,19.25,0.01,780.81,0.0
3,MViTv2\_B\_32x3,16,Infer,2,99.00,0.00,147.14,0.15,13.59,0.01,845.20,0.0
4,MViTv2\_S\_16x4,16,Infer,4,99.59,0.03,204.00,0.06,19.61,0.01,1412.77,0.0
5,MViTv2\_B\_32x3,16,Infer,4,99.51,0.04,288.79,0.06,13.85,0.00,1476.77,0.0
6,MViTv2\_S\_16x4,16,Infer,8,100.00,0.00,400.78,0.12,19.96,0.01,2679.05,0.0
7,MViTv2\_B\_32x3,16,Infer,8,100.00,0.00,567.22,0.08,14.10,0.00,2742.12,0.0
8,MViTv2\_S\_16x4,16,Infer,16,100.00,0.00,798.01,0.11,20.05,0.00,5208.50,0.0
9,MViTv2\_B\_32x3,16,Infer,16,100.00,0.00,1126.58,0.19,14.20,0.00,5272.07,0.0


### Prep for LaTeX 

In [4]:
# Drop standard deviation columns and rename for table output
mean_cols = {
    "model": "Model",
    "num_frames": "Frames",
    "batch_size": "BS",
    "gpu_util_mean": "GPU Util.",
    "latency_ms_mean": "Latency (ms)",
    "throughput_samp_per_s_mean": "Throughput (samp/s)",
    "peak_mem_mb_mean": "Peak Mem. (MB)",
}

train_mean_df = train_df[list(mean_cols.keys())].rename(columns=mean_cols)
infer_mean_df = infer_df[list(mean_cols.keys())].rename(columns=mean_cols)

def format_benchmark_df(df):
    df = df.copy()
    df["GPU Util."] = df["GPU Util."].apply(lambda x: f"{x:.2f}\\%")
    df["Latency (ms)"] = df["Latency (ms)"].apply(lambda x: f"{x:,.2f}")
    df["Throughput (samp/s)"] = df["Throughput (samp/s)"].apply(lambda x: f"{x:,.2f}")
    df["Peak Mem. (MB)"] = df["Peak Mem. (MB)"].apply(lambda x: f"{x:,.2f}")
    return df

train_mean_df = format_benchmark_df(train_mean_df)
infer_mean_df = format_benchmark_df(infer_mean_df)

display(train_mean_df)
display(infer_mean_df)

,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S\_16x4,16,1,98.81\%,224.06,4.46,"1,800.72"
1,MViTv2\_B\_32x3,16,1,98.73\%,319.51,3.13,"2,598.79"
2,MViTv2\_S\_16x4,16,2,99.56\%,430.89,4.64,"3,293.91"
3,MViTv2\_B\_32x3,16,2,99.40\%,613.38,3.26,"4,759.60"
4,MViTv2\_S\_16x4,16,4,99.83\%,828.60,4.83,"6,223.95"
5,MViTv2\_B\_32x3,16,4,99.83\%,"1,174.99",3.40,"9,070.12"
6,MViTv2\_S\_16x4\_e,32,1,99.75\%,574.10,1.74,"3,950.84"
7,MViTv2\_B\_32x3,32,1,99.72\%,807.67,1.24,"5,599.70"
8,MViTv2\_S\_16x4\_e,32,2,99.89\%,"1,124.22",1.78,"7,572.60"
9,MViTv2\_B\_32x3,32,2,99.86\%,"1,582.92",1.26,"10,733.43"


,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S\_16x4,16,1,98.00\%,54.16,18.46,468.83
1,MViTv2\_B\_32x3,16,1,98.00\%,76.25,13.11,532.39
2,MViTv2\_S\_16x4,16,2,99.00\%,103.88,19.25,780.81
3,MViTv2\_B\_32x3,16,2,99.00\%,147.14,13.59,845.20
4,MViTv2\_S\_16x4,16,4,99.59\%,204.00,19.61,"1,412.77"
5,MViTv2\_B\_32x3,16,4,99.51\%,288.79,13.85,"1,476.77"
6,MViTv2\_S\_16x4,16,8,100.00\%,400.78,19.96,"2,679.05"
7,MViTv2\_B\_32x3,16,8,100.00\%,567.22,14.10,"2,742.12"
8,MViTv2\_S\_16x4,16,16,100.00\%,798.01,20.05,"5,208.50"
9,MViTv2\_B\_32x3,16,16,100.00\%,"1,126.58",14.20,"5,272.07"


### Print LaTeX

In [5]:
def fixhlines(txt: str) -> str:
    return (
        txt.replace("\\toprule", "\\hline")
        .replace("\\midrule", "\\hline")
        .replace("\\bottomrule", "\\hline")
    )

In [6]:
train_latex = train_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Training benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:benchmark_train",
    position="ht",
)

infer_latex = infer_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Inference benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:benchmark_infer",
    position="ht",
)

print("Training Table LaTeX:")
print(fixhlines(train_latex))
print("\nInference Table LaTeX:")
print(fixhlines(infer_latex))

Training Table LaTeX:
\begin{table}[ht]
\caption{Training benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.}
\label{tab:benchmark_train}
\begin{tabular}{|l|c|c|r|r|r|r|}
\hline
Model & Frames & BS & GPU Util. & Latency (ms) & Throughput (samp/s) & Peak Mem. (MB) \\
\hline
MViTv2\_S\_16x4 & 16 & 1 & 98.81\% & 224.06 & 4.46 & 1,800.72 \\
MViTv2\_B\_32x3 & 16 & 1 & 98.73\% & 319.51 & 3.13 & 2,598.79 \\
MViTv2\_S\_16x4 & 16 & 2 & 99.56\% & 430.89 & 4.64 & 3,293.91 \\
MViTv2\_B\_32x3 & 16 & 2 & 99.40\% & 613.38 & 3.26 & 4,759.60 \\
MViTv2\_S\_16x4 & 16 & 4 & 99.83\% & 828.60 & 4.83 & 6,223.95 \\
MViTv2\_B\_32x3 & 16 & 4 & 99.83\% & 1,174.99 & 3.40 & 9,070.12 \\
MViTv2\_S\_16x4\_e & 32 & 1 & 99.75\% & 574.10 & 1.74 & 3,950.84 \\
MViTv2\_B\_32x3 & 32 & 1 & 99.72\% & 807.67 & 1.24 & 5,599.70 \\
MViTv2\_S\_16x4\_e & 32 & 2 & 99.89\% & 1,124.22 & 1.78 & 7,572.60 \\
MViTv2\_B\_32x3 & 32 & 2 & 99.86\% & 1,582.92 & 1.26 & 10,733.43 \\
\hline
\end{tabular}
\end{table}


Inference